# Improving Air Quality Prediction Using Bayesian Optimization For Feature Selection And Hyperparameter Tuning

**Objective:**

1. To compare the predictive performance of advanced machine learning algorithms, specifically Random Forest, XGBoost, CatBoost, and Support Vector Regression (SVR), in forecasting the Air Quality.

2. To improve prediction performance and generalization by using Bayesian Optimization for both feature selection and hyperparameter adjustment of the best-performing models identified in the comparative.

3. To develop and validate a Stacking Ensemble model that integrates optimized Random Forest, XGBoost, CatBoost, and SVR algorithms, aiming to achieve a quantitative benchmark of a coefficient of determination (R²) exceeding 0.80.

Kuching only. Predicts **PM2.5** from four pollutant inputs (**NO2, O3, CO, PM10**) multivariate, **no lag features** then derives AQI using the US EPA piecewise-linear breakpoint formula.

**Pipeline:** `preprocess → Bayesian feature selection → Training → Bayesian hyperparameter tuning 4 models → Stacking Ensemble → Testing → Performance Predict & Evaluation → AQI Forecast`

In [1]:
# Colab / environment setup
!pip install catboost scikit-optimize xgboost openpyxl -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Phase 1: Imports
import pandas as pd
import numpy as np
import warnings
import os
import joblib
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from skopt import BayesSearchCV, gp_minimize
from skopt.space import Real, Integer

print('All imports successful.')

All imports successful.


In [3]:
# Phase 2: Data Loading (Kuching DOE Malaysia)
dataset_path = 'Dataset Kuching/KuchingDataset.xlsx'

df = pd.read_excel(dataset_path, header=None,
                   names=['NO2', 'O3', 'CO', 'PM10', 'PM2.5'])
print(f'Raw dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Raw dataset shape: (3026, 5)
Columns: ['NO2', 'O3', 'CO', 'PM10', 'PM2.5']


,NO2,O3,CO,PM10,PM2.5
0,0.004,0.011,0.447,23.667,NaN
1,0.003,0.013,0.440,21.625,NaN
2,0.003,0.013,0.399,12.083,NaN
3,0.005,0.011,0.503,39.167,NaN
4,0.006,0.009,0.571,32.542,NaN


In [4]:
# Phase 3: Data Preprocessing (Cleaning, Normalization, Split)

# 3.1 Remove duplicates and coerce to numeric
df = df.drop_duplicates().reset_index(drop=True)
pollutant_cols = ['NO2', 'O3', 'CO', 'PM10', 'PM2.5']
for col in pollutant_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace(0, np.nan)

# 3.2 Drop rows where the TARGET (PM2.5) is missing — cannot learn from unlabelled samples
before = len(df)
df = df.dropna(subset=['PM2.5']).reset_index(drop=True)
print(f'Rows dropped (missing PM2.5) : {before - len(df)}')

# 3.3 Mean imputation for INPUT FEATURES only (NO2, O3, CO, PM10)
for col in ['NO2', 'O3', 'CO', 'PM10']:
    df[col] = df[col].fillna(df[col].mean())

print(f'Dataset shape after cleaning : {df.shape}')
print(f'Missing values remaining     : {df.isnull().sum().sum()}')

# 3.4 Define features and target
feature_cols = ['NO2', 'O3', 'CO', 'PM10']
X = df[feature_cols].copy()
y = df['PM2.5'].copy()

# 3.5 Chronological 80/20 split BEFORE scaling (prevents data leakage)
split_idx = int(len(X) * 0.8)
X_train_raw = X.iloc[:split_idx].reset_index(drop=True)
X_test_raw  = X.iloc[split_idx:].reset_index(drop=True)
y_train     = y.iloc[:split_idx].reset_index(drop=True)
y_test      = y.iloc[split_idx:].reset_index(drop=True)

# 3.6 Min-Max Normalization — PM10 only (fit on train only; transform both)
scaler = MinMaxScaler()
X_train = X_train_raw.copy()
X_test  = X_test_raw.copy()
X_train[['PM10']] = scaler.fit_transform(X_train_raw[['PM10']])
X_test[['PM10']]  = scaler.transform(X_test_raw[['PM10']])

print(f'Training set : {X_train.shape}  |  Testing set : {X_test.shape}')
print(f'Features ({len(feature_cols)}): {feature_cols}  (normalized: PM10 only)')
X_train.head()

Rows dropped (missing PM2.5) : 469
Dataset shape after cleaning : (2557, 5)
Missing values remaining     : 0
Training set : (2045, 4)  |  Testing set : (512, 4)
Features (4): ['NO2', 'O3', 'CO', 'PM10']  (normalized: PM10 only)


,NO2,O3,CO,PM10
0,0.0057,0.0083,0.694,0.006320
1,0.0068,0.0069,0.763,0.003670
2,0.0047,0.0098,0.784,0.006342
3,0.0057,0.0098,0.720,0.012041
4,0.0067,0.0055,0.746,0.008004


In [5]:
# Phase 4: Bayesian Feature Selection (Gaussian Process Minimization)
# Evaluates subsets of input features using cross-validated MSE (TimeSeriesSplit)

feature_names = X_train.columns.tolist()
tscv = TimeSeriesSplit(n_splits=5)

def feature_selection_objective(params):
    selected = [f for f, v in zip(feature_names, params) if v > 0.5]
    if not selected:
        return 9999.0
    scores = cross_val_score(
        RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        X_train[selected], y_train,
        cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1
    )
    return -scores.mean()

print('Running Bayesian feature selection (n_calls=30)...')
fs_result = gp_minimize(
    feature_selection_objective,
    [Real(0.0, 1.0, name=f) for f in feature_names],
    n_calls=30, random_state=42, acq_func='EI'
)

selected_features = [f for f, v in zip(feature_names, fs_result.x) if v > 0.5]
if len(selected_features) < 2:
    selected_features = feature_names
    print('Note: BO selected fewer than 2 features — retaining all features.')

X_train_sel = X_train[selected_features].copy()
X_test_sel  = X_test[selected_features].copy()

print(f'Selected features ({len(selected_features)}/{len(feature_names)}): {selected_features}')
print(f'Best CV score (neg-MSE): {fs_result.fun:.4f}')

Running Bayesian feature selection (n_calls=30)...
Note: BO selected fewer than 2 features — retaining all features.
Selected features (4/4): ['NO2', 'O3', 'CO', 'PM10']
Best CV score (neg-MSE): 73.9198


In [6]:
# Phase 5: Baseline Models — Default Hyperparameters (comparative benchmark)

def evaluate_model(model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    return {
        'RMSE': round(float(np.sqrt(mean_squared_error(y_te, preds))), 4),
        'MAE' : round(float(mean_absolute_error(y_te, preds)), 4),
        'R2'  : round(float(r2_score(y_te, preds)), 4),
    }

default_models = [
    ('Random Forest (Default)',
     RandomForestRegressor(random_state=42, n_jobs=-1)),
    ('XGBoost (Default)',
     XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)),
    ('CatBoost (Default)',
     CatBoostRegressor(verbose=0, random_state=42)),
    ('SVR (Default)',
     SVR()),
]

baseline_records = []
for name, model in default_models:
    r = evaluate_model(model, X_train_sel, X_test_sel, y_train, y_test)
    r['Model'] = name
    baseline_records.append(r)

baseline_df = pd.DataFrame(baseline_records)[['Model', 'RMSE', 'MAE', 'R2']]
print('Baseline Model Performance (Test Set):')
print(baseline_df.to_string(index=False))

Baseline Model Performance (Test Set):
                  Model   RMSE    MAE     R2
Random Forest (Default) 2.5323 1.7226 0.9229
      XGBoost (Default) 3.1470 2.0194 0.8809
     CatBoost (Default) 2.8716 1.8436 0.9008
          SVR (Default) 4.2868 2.4382 0.7790


In [7]:
# Phase 6: Bayesian Hyperparameter Tuning (gp_minimize + TimeSeriesSplit CV)
# Custom objective gives full control: degenerate folds (rare constant-target slices)
# are detected and skipped so CatBoost never receives an all-equal y array.

tscv5   = TimeSeriesSplit(n_splits=5)
BO_ITER = 30

def bo_tune(model_cls, space, fixed_kw):
    """Bayesian optimisation over model hyperparameters via gp_minimize.
    Scores via MAE averaged across TimeSeriesSplit folds; skips degenerate folds."""
    dim_names = [d.name for d in space]

    def objective(params):
        kw = {k: (int(v) if isinstance(v, np.integer) else
                  float(v) if isinstance(v, np.floating) else v)
              for k, v in zip(dim_names, params)}
        model = model_cls(**kw, **fixed_kw)
        fold_maes = []
        for tr_idx, va_idx in tscv5.split(X_train_sel):
            y_tr = y_train.iloc[tr_idx]
            if y_tr.nunique() < 2:      # skip fold if target is constant
                continue
            try:
                model.fit(X_train_sel.iloc[tr_idx], y_tr)
                fold_maes.append(
                    mean_absolute_error(y_train.iloc[va_idx],
                                        model.predict(X_train_sel.iloc[va_idx]))
                )
            except Exception:
                continue                # skip any other unexpected error
        return np.mean(fold_maes) if fold_maes else 9999.0

    res = gp_minimize(objective, space, n_calls=BO_ITER, random_state=42, acq_func='EI')
    return {k: (int(v) if isinstance(v, np.integer) else
                float(v) if isinstance(v, np.floating) else v)
            for k, v in zip(dim_names, res.x)}

# ── Random Forest ─────────────────────────────────────────────────
print('Tuning Random Forest...')
rf_params = bo_tune(
    RandomForestRegressor,
    [Integer(100, 500, name='n_estimators'),
     Integer(3,   20,  name='max_depth'),
     Integer(2,   10,  name='min_samples_split'),
     Real(0.3,  1.0,   name='max_features')],
    {'random_state': 42, 'n_jobs': -1}
)
print(f'  Best params: {rf_params}')
best_rf = RandomForestRegressor(**rf_params, random_state=42, n_jobs=-1)
best_rf.fit(X_train_sel, y_train)

# ── XGBoost ───────────────────────────────────────────────────────
print('Tuning XGBoost...')
xgb_params = bo_tune(
    XGBRegressor,
    [Integer(50,  500,  name='n_estimators'),
     Integer(3,   10,   name='max_depth'),
     Real(0.01, 0.3,    name='learning_rate',    prior='log-uniform'),
     Real(0.5,  1.0,    name='subsample'),
     Real(0.5,  1.0,    name='colsample_bytree'),
     Real(0.0,  1.0,    name='reg_alpha')],
    {'random_state': 42, 'objective': 'reg:squarederror', 'verbosity': 0}
)
print(f'  Best params: {xgb_params}')
best_xgb = XGBRegressor(**xgb_params, random_state=42,
                         objective='reg:squarederror', verbosity=0)
best_xgb.fit(X_train_sel, y_train)

# ── CatBoost ──────────────────────────────────────────────────────
print('Tuning CatBoost...')
cat_params = bo_tune(
    CatBoostRegressor,
    [Real(0.01, 0.3,    name='learning_rate', prior='log-uniform'),
     Integer(4,  10,    name='depth'),
     Integer(200, 1000, name='iterations'),
     Real(1.0,  10.0,   name='l2_leaf_reg')],
    {'verbose': 0, 'random_state': 42}
)
print(f'  Best params: {cat_params}')
best_cat = CatBoostRegressor(**cat_params, verbose=0, random_state=42)
best_cat.fit(X_train_sel, y_train)

# ── SVR ───────────────────────────────────────────────────────────
print('Tuning SVR...')
svr_params = bo_tune(
    SVR,
    [Real(0.1,  100.0, name='C',       prior='log-uniform'),
     Real(0.001, 1.0,  name='epsilon', prior='log-uniform'),
     Real(1e-4,  1.0,  name='gamma',   prior='log-uniform')],
    {}
)
print(f'  Best params: {svr_params}')
best_svr = SVR(**svr_params)
best_svr.fit(X_train_sel, y_train)

print('\nAll four models tuned and fitted.')

Tuning Random Forest...
  Best params: {'n_estimators': 118, 'max_depth': 6, 'min_samples_split': 2, 'max_features': 1.0}
Tuning XGBoost...
  Best params: {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.11323444248150018, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_alpha': 0.0}
Tuning CatBoost...
  Best params: {'learning_rate': 0.059149706073998436, 'depth': 4, 'iterations': 242, 'l2_leaf_reg': 1.0}
Tuning SVR...
  Best params: {'C': 100.0, 'epsilon': 0.001, 'gamma': 0.14466594019398776}

All four models tuned and fitted.


In [8]:
# Evaluate BO-Optimized Models on Test Set

optimized_records = []
for name, model in [
    ('Random Forest (Optimized)', best_rf),
    ('XGBoost (Optimized)',       best_xgb),
    ('CatBoost (Optimized)',      best_cat),
    ('SVR (Optimized)',           best_svr),
]:
    preds = model.predict(X_test_sel)
    optimized_records.append({
        'Model': name,
        'RMSE' : round(float(np.sqrt(mean_squared_error(y_test, preds))), 4),
        'MAE'  : round(float(mean_absolute_error(y_test, preds)), 4),
        'R2'   : round(float(r2_score(y_test, preds)), 4),
    })

optimized_df = pd.DataFrame(optimized_records)[['Model', 'RMSE', 'MAE', 'R2']]
print('BO-Optimized Model Performance (Test Set):')
print(optimized_df.to_string(index=False))

print('\nImprovement over default baseline (MAE):')
for b, o in zip(baseline_records, optimized_records):
    imp   = (b['MAE'] - o['MAE']) / b['MAE'] * 100
    sign  = '+' if imp > 0 else ''
    label = b['Model'].replace(' (Default)', '')
    print(f'  {label:15s}  Default={b["MAE"]:.4f}  Optimized={o["MAE"]:.4f}  ({sign}{imp:.2f}%)')

BO-Optimized Model Performance (Test Set):
                    Model   RMSE    MAE     R2
Random Forest (Optimized) 2.2208 1.4670 0.9407
      XGBoost (Optimized) 2.8687 1.8841 0.9010
     CatBoost (Optimized) 2.3555 1.5409 0.9333
          SVR (Optimized) 2.8100 1.7266 0.9051

Improvement over default baseline (MAE):
  Random Forest    Default=1.7226  Optimized=1.4670  (+14.84%)
  XGBoost          Default=2.0194  Optimized=1.8841  (+6.70%)
  CatBoost         Default=1.8436  Optimized=1.5409  (+16.42%)
  SVR              Default=2.4382  Optimized=1.7266  (+29.19%)


In [9]:
# Phase 7: Stacking Ensemble
# Base learners : RF + XGBoost + CatBoost + SVR  (all BO-optimized)
# Meta-learner  : XGBoost
# cv=5 uses KFold internally — required because StackingRegressor uses
# cross_val_predict, which needs every sample to appear in exactly one test fold.
# TimeSeriesSplit does NOT satisfy this and raises a ValueError.

estimators = [
    ('rf',  best_rf),
    ('xgb', best_xgb),
    ('cat', best_cat),
    ('svr', best_svr),
]

stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=XGBRegressor(
        n_estimators=200, learning_rate=0.05,
        max_depth=4, random_state=42, verbosity=0,
        objective='reg:squarederror'
    ),
    cv=5,           # KFold(5) — compatible with cross_val_predict
    passthrough=True,
    n_jobs=-1,
)

print('Training Stacking Ensemble (Meta-Learner)...')
stacking_model.fit(X_train_sel, y_train)
print('Stacking Ensemble training complete.')

Training Stacking Ensemble (Meta-Learner)...
Stacking Ensemble training complete.


In [10]:
# Phase 8: Training & Testing Phase — Full Performance Evaluation

all_models = [
    ('Random Forest (Optimized)',        best_rf),
    ('XGBoost (Optimized)',              best_xgb),
    ('CatBoost (Optimized)',             best_cat),
    ('SVR (Optimized)',                  best_svr),
    ('Stacking Ensemble (Meta-Learner)', stacking_model),
]

# ── Training Phase ────────────────────────────────────────────────
print('=' * 68)
print('TRAINING PHASE')
print('Training Set -> Bayesian Optimization -> Stacking Ensemble')
print('=' * 68)
train_records = []
for name, model in all_models:
    p = model.predict(X_train_sel)
    train_records.append({
        'Model'     : name,
        'Train RMSE': round(float(np.sqrt(mean_squared_error(y_train, p))), 4),
        'Train MAE' : round(float(mean_absolute_error(y_train, p)), 4),
        'Train R2'  : round(float(r2_score(y_train, p)), 4),
    })
print(pd.DataFrame(train_records).to_string(index=False))

# ── Testing Phase ─────────────────────────────────────────────────
print()
print('=' * 68)
print('TESTING PHASE')
print('Testing Set -> Performance Predict & Evaluation (R2, RMSE, MAE)')
print('=' * 68)
test_records = []
for name, model in all_models:
    p = model.predict(X_test_sel)
    test_records.append({
        'Model'    : name,
        'Test RMSE': round(float(np.sqrt(mean_squared_error(y_test, p))), 4),
        'Test MAE' : round(float(mean_absolute_error(y_test, p)), 4),
        'Test R2'  : round(float(r2_score(y_test, p)), 4),
    })
print(pd.DataFrame(test_records).to_string(index=False))

# ── Overfitting Check ─────────────────────────────────────────────
print()
print('=' * 68)
print('TRAIN vs TEST COMPARISON (Overfitting Check)')
print('=' * 68)
gap_records = [{
    'Model'    : tr['Model'],
    'Train MAE': tr['Train MAE'],
    'Test MAE' : te['Test MAE'],
    'Train R2' : tr['Train R2'],
    'Test R2'  : te['Test R2'],
    'MAE Gap'  : round(te['Test MAE'] - tr['Train MAE'], 4),
} for tr, te in zip(train_records, test_records)]
print(pd.DataFrame(gap_records).to_string(index=False))
print('Note: MAE Gap = Test MAE - Train MAE  (positive = generalises with some gap)')

# ── Stacking Ensemble Final Summary ──────────────────────────────
stack_tr = train_records[-1]
stack_te = test_records[-1]
print()
print('=' * 68)
print('STACKING ENSEMBLE -> FINAL PERFORMANCE SUMMARY')
print('=' * 68)
print(f'  Train  RMSE={stack_tr["Train RMSE"]:.4f}  MAE={stack_tr["Train MAE"]:.4f}  R2={stack_tr["Train R2"]:.4f}')
print(f'  Test   RMSE={stack_te["Test RMSE"]:.4f}  MAE={stack_te["Test MAE"]:.4f}  R2={stack_te["Test R2"]:.4f}')

# ── Objective Validation ──────────────────────────────────────────
avg_baseline_mae = baseline_df['MAE'].mean()
best_baseline    = baseline_df.loc[baseline_df['MAE'].idxmin()]
stack_mae        = stack_te['Test MAE']
mae_vs_avg       = (avg_baseline_mae - stack_mae) / avg_baseline_mae * 100
mae_vs_best      = (best_baseline['MAE'] - stack_mae) / best_baseline['MAE'] * 100
r2_pass          = stack_te['Test R2'] >= 0.80
mae_pass         = mae_vs_avg >= 15

print()
print('Objective Validation:')
print(f'  Avg baseline Test MAE       : {avg_baseline_mae:.4f}')
print(f'  Best baseline               : {best_baseline["Model"]}  (MAE={best_baseline["MAE"]:.4f})')
print(f'  Stacking Ensemble Test MAE  : {stack_mae:.4f}')
print(f'  MAE reduction vs avg (%)    : {mae_vs_avg:.2f}%')
print(f'  MAE reduction vs best (%)   : {mae_vs_best:.2f}%')
print(f'  R2 >= 0.80                  : {r2_pass}  ({stack_te["Test R2"]:.4f})')
print(f'  MAE reduction >= 15% target : {mae_pass}  ({mae_vs_avg:.2f}%)')

# Predictions and combined table for downstream cells
stack_preds = stacking_model.predict(X_test_sel)

all_results = pd.concat([
    baseline_df.rename(columns={'RMSE': 'Test RMSE', 'MAE': 'Test MAE', 'R2': 'Test R2'}),
    optimized_df.rename(columns={'RMSE': 'Test RMSE', 'MAE': 'Test MAE', 'R2': 'Test R2'}),
    pd.DataFrame([{
        'Model'    : 'Stacking Ensemble (Meta-Learner)',
        'Test RMSE': stack_te['Test RMSE'],
        'Test MAE' : stack_te['Test MAE'],
        'Test R2'  : stack_te['Test R2'],
    }]),
], ignore_index=True)

print()
print('Full Model Comparison (Baseline vs Optimized vs Stacking):')
print(all_results.to_string(index=False))

TRAINING PHASE
Training Set -> Bayesian Optimization -> Stacking Ensemble
                           Model  Train RMSE  Train MAE  Train R2
       Random Forest (Optimized)      1.6959     1.2629    0.9870
             XGBoost (Optimized)      1.2302     0.9317    0.9932
            CatBoost (Optimized)      1.7108     1.2629    0.9868
                 SVR (Optimized)      2.2872     1.5652    0.9764
Stacking Ensemble (Meta-Learner)      2.5691     1.3496    0.9702

TESTING PHASE
Testing Set -> Performance Predict & Evaluation (R2, RMSE, MAE)
                           Model  Test RMSE  Test MAE  Test R2
       Random Forest (Optimized)     2.2208    1.4670   0.9407
             XGBoost (Optimized)     2.8687    1.8841   0.9010
            CatBoost (Optimized)     2.3555    1.5409   0.9333
                 SVR (Optimized)     2.8100    1.7266   0.9051
Stacking Ensemble (Meta-Learner)     2.4534    1.5983   0.9276

TRAIN vs TEST COMPARISON (Overfitting Check)
                           

In [11]:
# Phase 8b: Train/Test Split Sensitivity Analysis (5 Runs)
# Repeats the full evaluation across five chronological split ratios
# (50/50, 60/40, 70/30, 80/20, 90/10) to test the robustness of the
# stacking ensemble to the amount of training data. The tuned
# hyperparameters from Phase 6 are REUSED (not re-tuned) so this run
# isolates the effect of the split ratio alone. Splitting stays
# chronological (no shuffling) — required for time-series data — and the
# scaler is re-fit on each run's training partition only (no leakage).

split_ratios = [0.50, 0.60, 0.70, 0.80, 0.90]
sensitivity_records = []

for tr_frac in split_ratios:
    si = int(len(X) * tr_frac)

    # Chronological re-split on the RAW (unscaled) features
    Xtr_raw = X.iloc[:si].reset_index(drop=True)
    Xte_raw = X.iloc[si:].reset_index(drop=True)
    ytr_r   = y.iloc[:si].reset_index(drop=True)
    yte_r   = y.iloc[si:].reset_index(drop=True)

    # Min-Max scaling fit on THIS run's training partition only
    sc = MinMaxScaler()
    Xtr_s = pd.DataFrame(sc.fit_transform(Xtr_raw), columns=feature_cols)[selected_features]
    Xte_s = pd.DataFrame(sc.transform(Xte_raw),     columns=feature_cols)[selected_features]

    # Rebuild base learners from the Phase-6 tuned params (no re-tuning)
    rf_r  = RandomForestRegressor(**rf_params, random_state=42, n_jobs=-1)
    xgb_r = XGBRegressor(**xgb_params, random_state=42,
                         objective='reg:squarederror', verbosity=0)
    cat_r = CatBoostRegressor(**cat_params, verbose=0, random_state=42)
    svr_r = SVR(**svr_params)
    for m in (rf_r, xgb_r, cat_r, svr_r):
        m.fit(Xtr_s, ytr_r)

    stack_r = StackingRegressor(
        estimators=[('rf', rf_r), ('xgb', xgb_r), ('cat', cat_r), ('svr', svr_r)],
        final_estimator=XGBRegressor(n_estimators=200, learning_rate=0.05,
                                     max_depth=4, random_state=42, verbosity=0,
                                     objective='reg:squarederror'),
        cv=5, passthrough=True, n_jobs=-1,
    )
    stack_r.fit(Xtr_s, ytr_r)

    ptr = stack_r.predict(Xtr_s)
    pte = stack_r.predict(Xte_s)
    sensitivity_records.append({
        'Split'    : f'{int(round(tr_frac*100))}/{int(round((1-tr_frac)*100))}',
        'Train n'  : len(ytr_r),
        'Test n'   : len(yte_r),
        'Train R2' : round(float(r2_score(ytr_r, ptr)), 4),
        'Test R2'  : round(float(r2_score(yte_r, pte)), 4),
        'Test RMSE': round(float(np.sqrt(mean_squared_error(yte_r, pte))), 4),
        'Test MAE' : round(float(mean_absolute_error(yte_r, pte)), 4),
    })
    print(f"Completed split {sensitivity_records[-1]['Split']}  ->  Test R2 = {sensitivity_records[-1]['Test R2']:.4f}")

sensitivity_df = pd.DataFrame(sensitivity_records)

print()
print('=' * 68)
print('TRAIN/TEST SPLIT SENSITIVITY — STACKING ENSEMBLE (5 RUNS)')
print('=' * 68)
print(sensitivity_df.to_string(index=False))

print()
print('Stacking Ensemble — mean +/- std across the 5 runs:')
print(f"  Test R2   : {sensitivity_df['Test R2'].mean():.4f}  +/-  {sensitivity_df['Test R2'].std():.4f}")
print(f"  Test RMSE : {sensitivity_df['Test RMSE'].mean():.4f}  +/-  {sensitivity_df['Test RMSE'].std():.4f}")
print(f"  Test MAE  : {sensitivity_df['Test MAE'].mean():.4f}  +/-  {sensitivity_df['Test MAE'].std():.4f}")
print(f"  All five runs satisfy Objective 3 (R2 > 0.80): {bool((sensitivity_df['Test R2'] > 0.80).all())}")

Completed split 50/50  ->  Test R2 = 0.8772
Completed split 60/40  ->  Test R2 = 0.9134
Completed split 70/30  ->  Test R2 = 0.9110
Completed split 80/20  ->  Test R2 = 0.9285
Completed split 90/10  ->  Test R2 = 0.9520

TRAIN/TEST SPLIT SENSITIVITY — STACKING ENSEMBLE (5 RUNS)
Split  Train n  Test n  Train R2  Test R2  Test RMSE  Test MAE
50/50     1278    1279    0.9867   0.8772     2.5357    1.9079
60/40     1534    1023    0.9833   0.9134     2.2726    1.5728
70/30     1789     768    0.9826   0.9110     2.4965    1.6726
80/20     2045     512    0.9772   0.9285     2.4381    1.5868
90/10     2301     256    0.9627   0.9520     1.3895    1.0562

Stacking Ensemble — mean +/- std across the 5 runs:
  Test R2   : 0.9164  +/-  0.0273
  Test RMSE : 2.2265  +/-  0.4785
  Test MAE  : 1.5593  +/-  0.3116
  All five runs satisfy Objective 3 (R2 > 0.80): True


In [12]:
# Phase 9: PM2.5 -> AQI Forecast (US EPA Piecewise-Linear Breakpoints)

AQI_BREAKPOINTS = [
    (0.0,    12.0,   0,   50,  'Good'),
    (12.1,   35.4,  51,  100,  'Moderate'),
    (35.5,   55.4, 101,  150,  'Unhealthy for Sensitive Groups'),
    (55.5,  150.4, 151,  200,  'Unhealthy'),
    (150.5, 250.4, 201,  300,  'Very Unhealthy'),
    (250.5, 350.4, 301,  400,  'Hazardous'),
    (350.5, 500.4, 401,  500,  'Hazardous'),
]

def pm25_to_aqi(pm25):
    pm25 = max(0.0, float(pm25))
    for c_lo, c_hi, a_lo, a_hi, _ in AQI_BREAKPOINTS:
        if c_lo <= pm25 <= c_hi:
            return round((a_hi - a_lo) / (c_hi - c_lo) * (pm25 - c_lo) + a_lo)
    return 500

def aqi_category(aqi):
    if aqi <=  50: return 'Good'
    if aqi <= 100: return 'Moderate'
    if aqi <= 150: return 'Unhealthy for Sensitive Groups'
    if aqi <= 200: return 'Unhealthy'
    if aqi <= 300: return 'Very Unhealthy'
    return 'Hazardous'

actual_aqi    = [pm25_to_aqi(v) for v in y_test.values]
predicted_aqi = [pm25_to_aqi(v) for v in stack_preds]

aqi_df = pd.DataFrame({
    'Actual_PM2.5'      : y_test.values,
    'Predicted_PM2.5'   : np.round(stack_preds, 3),
    'Actual_AQI'        : actual_aqi,
    'Predicted_AQI'     : predicted_aqi,
    'Actual_Category'   : [aqi_category(a) for a in actual_aqi],
    'Predicted_Category': [aqi_category(a) for a in predicted_aqi],
})

category_match = (aqi_df['Actual_Category'] == aqi_df['Predicted_Category']).mean() * 100

print('=== Accurate AQI Forecast — First 20 Samples ===')
print(aqi_df.head(20).to_string(index=False))
print(f'\nAQI Category Match Accuracy: {category_match:.2f}%')
print('\nPredicted AQI Category Distribution:')
print(aqi_df['Predicted_Category'].value_counts().to_string())

=== Accurate AQI Forecast — First 20 Samples ===
 Actual_PM2.5  Predicted_PM2.5  Actual_AQI  Predicted_AQI                Actual_Category             Predicted_Category
       23.916        23.193001          76             74                       Moderate                       Moderate
       20.429        13.135000          69             53                       Moderate                       Moderate
       28.511        26.580999          86             81                       Moderate                       Moderate
       21.763        16.888000          71             61                       Moderate                       Moderate
       19.575        14.243000          67             56                       Moderate                       Moderate
       26.684        21.858000          82             72                       Moderate                       Moderate
       33.275        31.260000          96             91                       Moderate                       

In [13]:
# Phase 10: Save Model Artifacts

save_dir = 'saved_model'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(stacking_model,    os.path.join(save_dir, 'stacking_model.pkl'))
joblib.dump(scaler,            os.path.join(save_dir, 'scaler.pkl'))
joblib.dump(selected_features, os.path.join(save_dir, 'all_features.pkl'))
joblib.dump(selected_features, os.path.join(save_dir, 'selected_features.pkl'))

print(f'Saved to {save_dir}/')
print(f'Features ({len(selected_features)}): {selected_features}')
print(f'Stacking Ensemble  Test R2={stack_te["Test R2"]:.4f}  RMSE={stack_te["Test RMSE"]:.4f}  MAE={stack_te["Test MAE"]:.4f}')

Saved to saved_model/
Features (4): ['NO2', 'O3', 'CO', 'PM10']
Stacking Ensemble  Test R2=0.9276  RMSE=2.4534  MAE=1.5983


In [14]:
# Phase 11: Export Results to Excel

output_file = 'Kuching_model_results.xlsx'

# Comprehensive train + test table (5 optimized models + stacking)
comprehensive_df = pd.DataFrame([{
    'Model'     : tr['Model'],
    'Train RMSE': tr['Train RMSE'],
    'Train MAE' : tr['Train MAE'],
    'Train R2'  : tr['Train R2'],
    'Test RMSE' : te['Test RMSE'],
    'Test MAE'  : te['Test MAE'],
    'Test R2'   : te['Test R2'],
    'MAE Gap'   : round(te['Test MAE'] - tr['Train MAE'], 4),
} for tr, te in zip(train_records, test_records)])

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # 1. Full comparison (baseline + optimized + stacking, test metrics)
    all_results.to_excel(writer, sheet_name='Model Comparison', index=False)
    # 2. Train vs Test for optimized + stacking
    comprehensive_df.to_excel(writer, sheet_name='Train vs Test', index=False)
    # 3. Baseline models (test only)
    baseline_df.to_excel(writer, sheet_name='Baseline Results', index=False)
    # 4. Stacked predictions
    pd.DataFrame({
        'Actual_PM25'            : y_test.values,
        'Stacking_Predicted_PM25': np.round(stack_preds, 3),
    }).to_excel(writer, sheet_name='Stacked Predictions', index=False)
    # 5. AQI forecast
    aqi_df.to_excel(writer, sheet_name='AQI Forecast', index=False)
    # 6. Objective validation summary
    pd.DataFrame([{
        'Stacking Test R2'           : stack_te['Test R2'],
        'Stacking Test RMSE'         : stack_te['Test RMSE'],
        'Stacking Test MAE'          : stack_te['Test MAE'],
        'Stacking Train R2'          : stack_tr['Train R2'],
        'Stacking Train RMSE'        : stack_tr['Train RMSE'],
        'Stacking Train MAE'         : stack_tr['Train MAE'],
        'AQI Category Match (%)'     : round(category_match, 2),
        'Avg Baseline MAE'           : round(avg_baseline_mae, 4),
        'MAE Reduction vs Avg (%)'   : round(mae_vs_avg, 2),
        'MAE Reduction vs Best (%)'  : round(mae_vs_best, 2),
        'R2 >= 0.80 Target Met'      : r2_pass,
        'MAE Reduction >= 15% Target': mae_pass,
        'Selected Features'          : str(selected_features),
    }]).to_excel(writer, sheet_name='Objective Validation', index=False)

print(f'Results saved to {output_file}')
print('Sheets: Model Comparison | Train vs Test | Baseline Results | Stacked Predictions | AQI Forecast | Objective Validation')

Results saved to Kuching_model_results.xlsx
Sheets: Model Comparison | Train vs Test | Baseline Results | Stacked Predictions | AQI Forecast | Objective Validation


In [15]:
# Baseline Model Performance — TRAINING SET (default hyperparameters)
# Produces the baseline train metrics needed for the Section 4.6 train tables.

baseline_train_records = []
for name, model in default_models:
    model.fit(X_train_sel, y_train)          # fit on training set
    p = model.predict(X_train_sel)           # predict on same training set
    baseline_train_records.append({
        'Model'     : name.replace(' (Default)', ''),
        'Train RMSE': round(float(np.sqrt(mean_squared_error(y_train, p))), 4),
        'Train MAE' : round(float(mean_absolute_error(y_train, p)), 4),
        'Train R2'  : round(float(r2_score(y_train, p)), 4),
    })

baseline_train_df = pd.DataFrame(baseline_train_records)[['Model', 'Train RMSE', 'Train MAE', 'Train R2']]
print('Baseline Model Performance (Training Set):')
print(baseline_train_df.to_string(index=False))

Baseline Model Performance (Training Set):
        Model  Train RMSE  Train MAE  Train R2
Random Forest      0.8277     0.5345    0.9969
      XGBoost      0.6198     0.4428    0.9983
     CatBoost      1.1685     0.8997    0.9938
          SVR     11.3489     2.8741    0.4179
